# Speaking Coach: fine-tune the grammar model on Colab (GPU)

Same training as `training/train_grammar.py` on your laptop, but on Colab's free T4 GPU (~50x faster).

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**
2. In Google Drive, create the folder `MyDrive/speaking_coach/` and upload into it:
   - `training/train_grammar.py`
   - `data/grammar/train_speech.tsv` and `data/grammar/dev_speech.tsv`
3. Run the cells top to bottom. Checkpoints are saved to Drive every 15 minutes, so if Colab disconnects, run everything again and training **resumes**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/speaking_coach'
!ls $DIR

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install sentencepiece

## Train
**Option A (recommended): bigger model, flan-t5-large (3x bigger than our current t5-base)**
- `--optim adafactor`: uses much less memory, so the large model fits on the free T4 (16 GB)
- `--lr 3e-4`: the usual learning rate for Adafactor with T5
- about 2-3 hours for 3 epochs. Checkpoints go to Drive every 15 min, so if Colab disconnects, run the cells again and it resumes.

**Option B: same t5-base, more epochs** (cell after this one)

In [ ]:
%cd $DIR
!python train_grammar.py --base google/flan-t5-large --optim adafactor --lr 3e-4 --train train_speech.tsv --dev dev_speech.tsv --out models/grammar-speech-large-v1 --epochs 3 --batch 16 --save-minutes 15

In [ ]:
# Option B (skip if you ran Option A)
%cd $DIR
!python train_grammar.py --train train_speech.tsv --dev dev_speech.tsv --out models/grammar-speech-v2 --epochs 3 --batch 32 --lr 5e-5 --save-minutes 15

## Quick test

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
path = f'{DIR}/models/grammar-speech-large-v1'
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSeq2SeqLM.from_pretrained(path).to('cuda')
for s in ['I studying in Gachon University.', 'So this was the last parts of the audio.',
          "I couldn't stop myself to generate my book."]:
    ids = tok('grammar: ' + s, return_tensors='pt').input_ids.to('cuda')
    print(s, '->', tok.decode(model.generate(ids, max_length=128, num_beams=5)[0], skip_special_tokens=True))

## Bring it home
Download the folder `MyDrive/speaking_coach/models/grammar-speech-large-v1` (or `grammar-speech-v2`) into `Speaking_Coach/models/` on your laptop, then measure it:
```
python -m evaluate.speech_eval models/grammar-speech-large-v1
python -m evaluate.grammar_eval 300 models/grammar-speech-large-v1
```